# 05 - Experience D : stéganalyse multi-classe (attribution d'algorithme)

Exploration. Jusqu'ici la question était binaire, image vierge ou porteuse. Ici on demande plus :
quel algorithme d'insertion a été utilisé. C'est une tâche multi-classe, l'attribution d'algorithme.

On travaille sur la source naturelle, avec quatre classes : cover, LSB, S-UNIWARD, HILL, sur
caractéristiques SRM déjà extraites. Deux volets :

1. un classifieur multinomial et sa matrice de confusion, qui donne le résultat ;
2. le lien avec S-SELECT : régler le k du k-means sur le nombre de classes, et voir pourquoi le critère
   non supervisé échoue ici alors que la variante supervisée tient.

Figures nommées expD_srm_<type>_natural_p<charge>.png.

## Configuration

In [ ]:
# ================= CONFIGURATION =================
FEATURE   = 'srm'
SOURCE    = 'natural'
CLASSES   = ['cover', 'lsb', 'uniward', 'hill']   # ordre des classes
PAYLOAD   = 0.4
N_PCA     = 300
SEED      = 42
# ================================================
pp = str(PAYLOAD).replace('.', '')
print('Experience D sur', FEATURE, '| source', SOURCE, '| classes', CLASSES)

In [ ]:
import os
import numpy as np
np.random.seed(SEED)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')
FEAT_DIR = f'{DATA_DIR}/features_{FEATURE}'
FIG = f'{DATA_DIR}/figures'
os.makedirs(FIG, exist_ok=True)
print('Caracteristiques :', FEAT_DIR, '| Figures :', FIG)

In [ ]:
!pip install -q scikit-learn matplotlib
print('Installation terminee.')

## 1. Chargement des quatre classes

Chaque classe est un fichier de caractéristiques. Le cover, la version LSB, la version S-UNIWARD et la
version HILL correspondent aux mêmes images de base. On équilibre les classes à la taille de la plus
petite pour ne pas biaiser le classifieur.

In [ ]:
def charger(setname):
    # simple precision : deux fois moins de memoire, indispensable a quatre classes sur SRM
    return np.load(f'{FEAT_DIR}/{SOURCE}__{setname}.npy').astype(np.float32)

blocs = {}
blocs['cover'] = charger('cover')
for a in ['lsb', 'uniward', 'hill']:
    blocs[a] = charger(f'{a}_p{PAYLOAD}')

n = min(len(blocs[c]) for c in CLASSES)
X = np.vstack([blocs[c][:n] for c in CLASSES]).astype(np.float32)
y = np.concatenate([[i] * n for i, c in enumerate(CLASSES)])
del blocs
print('images par classe :', n, '| total :', len(y), '| dimensions :', X.shape[1])

## 2. Classifieur multinomial et matrice de confusion

Même chaîne que dans les autres expériences, mise à l'échelle, réduction PCA, régression logistique, mais
en mode multinomial. On évalue par validation croisée, et on construit la matrice de confusion à partir de
prédictions hors échantillon, pour qu'elle soit honnête. La matrice est normalisée par ligne : chaque
ligne montre, pour une vraie classe, comment elle est répartie entre les classes prédites.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def classifieur():
    return make_pipeline(StandardScaler(copy=False),
                         PCA(n_components=N_PCA, svd_solver='randomized', random_state=SEED),
                         LogisticRegression(max_iter=5000))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
y_pred = cross_val_predict(classifieur(), X, y, cv=cv)

acc = accuracy_score(y, y_pred)
f1m = f1_score(y, y_pred, average='macro')
print(f'exactitude globale : {acc:.3f}')
print(f'F1 macro           : {f1m:.3f}')
print()
print(classification_report(y, y_pred, target_names=CLASSES, digits=3))

C = confusion_matrix(y, y_pred, normalize='true')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(C, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('classe predite'); ax.set_ylabel('vraie classe')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f'{C[i, j]:.2f}', ha='center', va='center',
                color='white' if C[i, j] > 0.5 else 'black')
ax.set_title(f'Matrice de confusion, {SOURCE}, {PAYLOAD} bpp, SRM')
fig.colorbar(im, fraction=0.046, pad=0.04)
nom = f'{FIG}/expD_srm_confusion_{SOURCE}_p{pp}.png'
plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
print('figure :', os.path.basename(nom))

## 3. Le lien avec S-SELECT

Ton intuition : le k du k-means de S-SELECT doit valoir le nombre de classes réelles. Attention, il y a
deux k-means dans la méthode. Celui de PFA regroupe les caractéristiques, son k est le nombre de
caractéristiques gardées. Celui du critère S-SELECT regroupe les images, et c'est lui qu'on règle sur le
nombre de classes, ici quatre.

On teste donc deux choses. D'abord le critère non supervisé, un k-means à k égal quatre sur les images,
mesuré par silhouette et par accord avec les vraies classes. Ensuite la sélection supervisée, qui garde
PFA pour enlever la redondance mais choisit les caractéristiques par leur performance de classification
multi-classe.

In [ ]:
# PFA compact et autonome, meme logique que src/pfa.py, composantes plafonnees pour rester rapide
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import cross_val_score

def pfa_indices(Xmat, n_feats, q=200, seed=SEED):
    Z = StandardScaler().fit_transform(Xmat)
    comp = PCA(n_components=min(q, Xmat.shape[0], Xmat.shape[1]), svd_solver='randomized', random_state=seed).fit(Z)
    A = comp.components_.T                      # chaque caracteristique par ses charges
    km = KMeans(n_clusters=n_feats, random_state=seed, n_init=10).fit(A)
    idx = []
    for c in range(n_feats):
        membres = np.where(km.labels_ == c)[0]
        d = np.linalg.norm(A[membres] - km.cluster_centers_[c], axis=1)
        idx.append(membres[np.argmin(d)])
    return np.array(sorted(idx))

# sous-echantillon pour garder le volet rapide
m = min(600, n)
sel = np.concatenate([np.where(y == i)[0][:m] for i in range(len(CLASSES))])
Xs, ys = X[sel], y[sel]

In [ ]:
# critere non supervise : k-means a k = nombre de classes sur les images
k = len(CLASSES)
Zs = StandardScaler().fit_transform(Xs)
Zs = PCA(n_components=N_PCA, random_state=SEED).fit_transform(Zs)
labels_ns = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(Zs)
print('critere non supervise, k =', k)
print(f'  silhouette                : {silhouette_score(Zs, labels_ns):.3f}')
print(f'  accord avec vraies classes (ARI) : {adjusted_rand_score(ys, labels_ns):.3f}')
print('  un ARI proche de 0 signifie que le regroupement ne retrouve pas les classes')

In [ ]:
# selection supervisee multi-classe : PFA propose, la performance de classification tranche
def score_multi(idx):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    return cross_val_score(pipe, Xs[:, idx], ys, cv=3, scoring='accuracy').mean()

print('reference, toutes caracteristiques (PCA 300) :',
      f'{cross_val_score(classifieur(), Xs, ys, cv=3, scoring="accuracy").mean():.3f}')
for nf in [24, 50, 100]:
    idx = pfa_indices(Xs, nf)
    print(f'PFA + supervise, {nf:3d} caracteristiques : {score_multi(idx):.3f}')

## Suite

On copie la matrice de confusion dans results et on consigne les chiffres dans results/experience_D.md,
avec l'interprétation dans results/figures_interpretations.md. Lecture attendue : LSB bien identifié, cover
correctement séparé, mais confusion probable entre S-UNIWARD et HILL, deux adaptatifs proches. Le critère
non supervisé devrait échouer, car les versions stégo d'une image restent collées à leur cover ; c'est ce
qui justifie la variante supervisée pour le multi-classe.